In [ ]:
import yfinance as yf
import schedule
import requests
import time
def get_stock_price(ticker):
 stock = yf.Ticker(ticker)
 price = stock.fast_info["last_price"]
 return price
print(get_stock_price("AAPL"))
TOKEN = "8349741044:AAH23anZDKiRtTgYwuVYQFBvCO_0QSkj72I"
chat_id = 1048880809
def send_alert(message):
   url = f"https://api.telegram.org/bot{TOKEN}/sendMessage"
   requests.post(url, data={"chat_id": chat_id, "text": message})

# define your thresholds
ALERTS = {
    "AAPL":    {"above": 210, "below": 180},
    "bitcoin": {"above": 70000, "below": 55000},
}

alerted = set()  # keeps track of what we already sent, so no spam

def check_prices():
    for asset, thresholds in ALERTS.items():
        if asset.isupper():  # stocks are uppercase like AAPL
            price = get_stock_price(asset)
        else:                # crypto like bitcoin
            price = get_crypto_price(asset)

        if thresholds["above"] and price > thresholds["above"]:
            key = f"{asset}_above"
            if key not in alerted:
                send_alert(f"{asset} is at ${price:.2f} — above your target!")
                alerted.add(key)

        if thresholds["below"] and price < thresholds["below"]:
            key = f"{asset}_below"
            if key not in alerted:
                send_alert(f"{asset} dropped to ${price:.2f} — below your floor!")
                alerted.add(key)

def get_crypto_price(coin_id):
    url = "https://api.coingecko.com/api/v3/simple/price"
    params = {"ids": coin_id, "vs_currencies": "usd"}
    r = requests.get(url, params=params)
    return r.json()[coin_id]["usd"]

def daily_digest():
    lines = ["Good morning! Your portfolio:\n"]
    for asset in ALERTS:
        if asset.isupper():
            price = get_stock_price(asset)
        else:
            price = get_crypto_price(asset)
        lines.append(f"  {asset}: ${price:,.2f}")
    send_alert("\n".join(lines))

# schedule everything
schedule.every(5).minutes.do(check_prices)
schedule.every().day.at("09:00").do(daily_digest)

print("Bot is running...")

while True:
    schedule.run_pending()
    time.sleep(30)




ModuleNotFoundError: No module named 'schedule'